In [2]:
import sys
import os

parent_path = os.path.abspath("..")  # Get absolute path of parent directory
if parent_path not in sys.path:  
    sys.path.append(parent_path)  # Add only if it's not already there
    
from dotenv import load_dotenv
load_dotenv("../.env")


import pandas as pd
import chromadb
import openai

from models.chromadb_functions import *  

### Read Excel file 

In [9]:
question_answer_df = pd.read_excel("../data/question_answer/100 câu hỏi.xlsx")

print(question_answer_df)

print("Excel Length: ", len(question_answer_df))

    Mã câu hỏi                                            Câu Hỏi  \
0            1  Hướng dẫn cho tôi cách sử dụng phân bón của Ph...   
1            2  Hướng dẫn cho tôi cách bón và liều lượng bón c...   
2            3  Hướng dẫn cho tôi cách bón và liều lượng bón c...   
3            4  Hướng dẫn cho tôi cách bón và liều lượng bón c...   
4            5  Hướng dẫn cho tôi cách bón và liều lượng bón c...   
..         ...                                                ...   
95          96  Bón phân cho cây lúa vào mùa mưa và mùa khô có...   
96          97  Khi nào là thời điểm bón phân tốt nhất cho cây...   
97          98         Có nên bón phân khi cây đang ra hoa không?   
98          99  Ngay sau khi thu hoạch cần phải bón phân liền ...   
99         100  Có thể bón phân trong giai đoạn cây bị sâu bện...   

                                          Câu trả lời  
0   Dạ. bà con vui lòng nói tên sản phẩm cụ thể hơ...  
1   Cách hướng dẫn sử dụng phân bón u rê hạt đục n...  
2  

### Get ChromaDB Collection

In [3]:
collection_path     = "vector_database"
collection_name     = os.getenv("CHROMADB_COLLECTION_NAME")
embedding_func      = define_embedding_function( api_key = os.getenv("OPENAI_API_KEY"), model_name = os.getenv("EMBEDDING_MODEL"))
similarity_method   = os.getenv("SIMILARITY_METHOD")

In [4]:
chromadb_collection = get_chroma_collection(collection_path, collection_name, embedding_func, similarity_method)

### Load Questions into ChromaDB
- With index in metadata and index for tracking

In [10]:
list_base_question  = list(question_answer_df["Câu Hỏi"])
list_question_no    = list(question_answer_df["Mã câu hỏi"])

In [11]:
chromadb_collection.add(
    documents=list_base_question,
    metadatas = [{"video": f"{i}.mp4"} for i in list_question_no],
    ids = [str(i) for i in list_question_no]
)

### Update question in ChromaDB using index

In [ ]:
chromadb_collection.update(
    ids=[str(i) for i in list_question_no],
    metadatas=[{"video": f"{i}.mp4"} for i in list_question_no]
)

### Query Similar Question in ChromaDB

In [ ]:
results = chromadb_collection.query(
    query_texts=["Làm sao để chăm sóc cây ăn trái?"], # Chroma will embed this for you
    n_results=1 # how many results to return
)

results["metadatas"]

{'ids': [['2a578124-9226-4252-82de-42fe8b6c7712']],
 'embeddings': None,
 'documents': [['ok']],
 'uris': None,
 'data': None,
 'metadatas': [[{'id': '1'}]],
 'distances': [[0.9252893222701347]],
 'included': [<IncludeEnum.distances: 'distances'>,
  <IncludeEnum.documents: 'documents'>,
  <IncludeEnum.metadatas: 'metadatas'>]}

In [30]:
results["metadatas"][0][0]["id"]

'1'

### Get all Data from Collection

In [13]:
data = chromadb_collection.get()
data

{'ids': ['dd2828c0-425b-4dba-847e-4e13cb849771',
  '2a578124-9226-4252-82de-42fe8b6c7712'],
 'embeddings': None,
 'documents': ['ok', 'ok'],
 'uris': None,
 'data': None,
 'metadatas': [{'id': '1'}, {'id': '1'}],
 'included': [<IncludeEnum.documents: 'documents'>,
  <IncludeEnum.metadatas: 'metadatas'>]}

dict

### Delete Data in ChromaDB

In [32]:
chromadb_collection.delete(
    ids=chromadb_collection.get()["ids"]
)


In [15]:
import uuid

chromadb_collection.add(
    documents=["ok"],
    metadatas = [{"id": "4"}],
    ids = [str(uuid.uuid4())]
)

In [10]:
a = str(uuid.uuid4())
a

'4b7e7279-32f6-4137-9560-68db94d10d73'

In [27]:
id = chromadb_collection.get(where={"id": "4"})
id['ids'][0]

'7e5caf72-2911-4cf3-8e3e-0f7ca6fe0870'

In [25]:
if len(id['ids']) ==0:
    print("ok")

ok
